# Reliability analysis

reliability is the property of an experimental study to yield similar results under different experimental conditions.
Let $\mathcal X$ be the space of results, $P$ the distribution of results, $\mathbf X, \mathbf X' \sim P^n$, and define $\hat P_{\mathbf X}$ as the empirical distribution of the (random) sample $\mathbf X$.
Define, for semplicity, the random variable $\text{MMD}_n := \text{MMD}(\hat P_{\mathbf X}, \hat P_{\mathbf X'})$.
Then, reliability is
\begin{equation}
    P^n \otimes P^n(\text{MMD}_n \leq \varepsilon),
\end{equation}
i.e., the probability that the empirical distributions of two samples of results are similar.

Now, one can relate the $\alpha$-quantile of $\text{MMD}_n$ ($q_\alpha(n)$) and $n$ with the following equation:
\begin{equation}
    \log(n) \approx -2 \log(q_n^\alpha) + \beta,
\end{equation}
where $\beta$ is a constant that depends on the kernel (for the MMD) and $P$.

Using the equation above and fixing $\alpha^*$ and $\varepsilon^*$, interpreted as a condition on how strict the reliability requirements are, one easily gets an estimate for the number of experiments $n^*$ that guarantees reliable results.

This notebook shows how to do that in a few steps, as well as to reproduce the experimental results of the paper.

## The experimental results file

The experimental results should be stored in a pandas-readable parquet file. 
In particular, the schema of the results file has to contain 
1. The experimental factors (variables influencing the outcome of an experiment).
2. A column for the alternatives (the benchmarked objects/methods).
3. A column for the result of the evaluation of a single alternative under a single combination of factors (the target column).

⚠️
If multiple scoring/metrics are used in the experiments, this should be included in an appropriate factor (called, for instance, `scoring`). 
Moreover, the metrics should either be all scores (such as ` sklearn.metrics.accuracy_score`) or all errors (such as ` sklearn.metrics.mean_squared_error`). 
At the moment, it is not possible to mix scores and errors due to the way rankings of the alternatives are constructed in `genexpy.utils.rankings.get_matrix_from_df`. 

After having a result file, ew can proceed to configure the reliability analysis.

## The configuration file

The first thing we need to do is to fill in the configuration file, `config.yaml`. 

In particular, we need to specify: 
- The path to the file containing the experimental results, `data.dataset_path`
- The description of the columns of the results file:
    - Indicate the factors and their kind (held-constant, design, reliability).
    - Indicate the name fo the column of alternatives, the target column, and whether the target is a score or an error.
- The parameters for reliability
    - The kernels (for the MMD)   
    - The thresholds $\alpha$ and $\delta$

### Classification of experimental factors
Held-constant factors are fixed to the indicated level. All other levels are ignored.
Each combination of design factors (called a configuration) will have its own, independent, reliability analysis.
We want to get results that are generalizable w.r.t. the reliability factors.

### Relationship between $\delta$ and $\varepsilon$
The code internally uses a parameter $\varepsilon$ to determine when two experimental results are ``close'' together. 
As fixing this threshold is not trivial, we make it depend on another parameter, $\delta$. 
The interpretation of $\delta$ and the way it and $\varepsilon$ relate is kernel-specific.
More details are in Section 4.2 of the paper.

## Running the analysis

Running the analysis is extremely simple. 
First, initialize a `ProjectManager` object with the path to the configuration file. The ProjectManager will load the configuration file, load the experimental results, initialize the internal parameters, and create the project structure within the parent directory of this file. 

`ProjectManager().reliability_analysis` will run the analysis.
The output is a pandas dataframe containing the predicted $n^*$ for all given configurations and thresholds. 
The method will also store the resampled distributions of the MMD and coefficients for the approximation of the ICDF of the MMD.

In [ ]:
# import os
# from genexpy.managers import ProjectManager
#
# main = ProjectManager("config.yaml", demo_dir=os.getcwd())
# df_nstar = main.reliability_analysis()

## Analysis of extremal cases

We analyze the simplest and hardest configurations in the study, namely, the ones which require respectively the smallest and greatest $n^*$ to achieve $(\alpha^*, \delta^*)$-reliability.


In [ ]:
# import pandas as pd
#
# alpha = 0.95
# delta = 0.05
# kernel = "JaccardKernel(k=1)"
# method = "embedding"
#
# dftmp = df_nstar.query("alpha == @alpha and delta == @delta and kernel == @kernel and method == @method")
# dftmp = dftmp.groupby(main.configuration_factors + ["N"])["nstar"].mean().reset_index()
# Nmax = dftmp.groupby(main.configuration_factors)["N"].max()
# dftmp = pd.merge(dftmp, Nmax, on=main.configuration_factors, suffixes=("", "_max"))
# dftmp = dftmp.query("N == N_max")
# dftmp

## Plots

To reproduce the plots in the paper, we initialize a `PlotManager` object and call its methods.
Upon initialization, the PlotManager will load the configuration file as well as the files created while running the ProjectManager (precomputed MMD and ICDF coefficients).

In [ ]:
import os
from genexpy.managers import PlotManager

plotter = PlotManager(config_yaml_path="config.yaml", demo_dir=os.getcwd())

The first set of plots shows the dependence of reliability on the desired reliability $\alpha^*$ and the similarity threshold $\delta^*$.
The parameter not shown in the axis is fixed to $\alpha^*=0.95, \delta^*=0.05$.

In [ ]:
plotter.plot_nstar_on_alpha_delta(alpha_fixed=0.95, delta_fixed=0.05)

The second set of plots is the simulated experimental study for a specific configuration.
We run $N$ experiments (i.e., sample $N$ results from the experimental results), estimate the reliability (top), and run a power-law regression to find $n^*_N$ (bottom).

In [ ]:
configuration = {"model": "LR", "tuning": "no", "scoring": "AUC"}
kernels = [plotter.kernels[2]]
plotter.plot_simulated_experimental_study(configuration=configuration, alpha=0.95, delta=0.05, kernels=kernels)

The third set of plots shows the 10-reliability achieved by the different configurations.

In [ ]:
import numpy as np
import pandas as pd
from genexpy.kernels.base import Kernel

n = 10

df = plotter.dfmmd.query("n == @n")
df.loc[:, "kernel"] = df["kernel"].replace("JaccardKernel(k=1)", "JaccardKernel(t=1)")

fixed_factors = ["model", "tuning", "scoring"]
groupby_keys = fixed_factors + ["kernel"]

df = df.merge(df.groupby(groupby_keys)["N"].max().reset_index(), on=groupby_keys, suffixes=["", "max"])
df = df.loc[df["N"] == df["Nmax"]]

deltas = [0.01, 0.05, 0.1, 0.2, 0.3]

def pleqeps(x, eps):
    return np.mean(x <= eps)

out = []
outt = []
for kernelname in df.kernel.unique():
    kernel = Kernel.from_string(kernelname)

    epss = np.array([kernel.get_eps(delta, na=plotter.na) for delta in deltas])

    tmp = df.query("kernel == @kernelname")

    tmp2 = tmp.groupby(groupby_keys)["mmd"].aggregate([lambda x: pleqeps(x, eps) for eps in epss])
    tmp2.columns = [f"rel_{n}(P, delta={delta:.2f})" for delta in deltas]
    tmp2 = tmp2.reset_index()

    out.append(tmp2)



dfrel = pd.concat(out, axis=0)

In [ ]:
da = pd.DataFrame({delta: dfrel[f"rel_{n}(P, delta={delta:.2f})"] for delta in deltas}).melt()
da

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 1)

sns.boxplot(data=da, x="variable", y="value", ax=ax)


